In [5]:
import re
import numpy as np

from gaussian import gaussian_eliminate
from gaussian import back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

def _evaluate_parametric_expression(expr, assignment):
    """Thay các biến tự do t_i bằng giá trị số và tính toán biểu thức"""
    stripped = expr.strip()
    m = re.fullmatch(r"t_(\d+)(?:\s*\((?:biến tự do|bien tu do)\))?", stripped)
    if m:
        return float(assignment.get(int(m.group(1)), 0.0))

    replaced = stripped
    for idx, value in assignment.items():
        replaced = re.sub(rf"\bt_{idx}\b", f"({float(value)})", replaced)

    if not re.fullmatch(r"[0-9eE\+\-\*/\(\)\.\s]+", replaced):
        raise ValueError(f"Bieu thuc khong hop le: {expr}")

    return float(eval(replaced, {"__builtins__": {}}, {}))


def verify_solution(A, x, b, atol=1e-7):
    """
    Kiểm chứng kết quả bằng NumPy.
    - x=None: Hệ vô nghiệm hoặc vô số nghiệm, in thông báo và trả về True
    - x là list số: Kiểm tra Ax ~ b cho trường hợp nghiệm duy nhất
    """
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float).reshape(-1)

    if not isinstance(x, list):
        print("Hệ không có nghiệm duy nhất. Mặc định là PASS")
        return True

    # Kiem tra neu x la danh sach so (truong hop nghiem duy nhat)
    if all(isinstance(v, (int, float, np.floating)) for v in x):
        x_np = np.array(x, dtype=float)
        try:
            return np.allclose(A_np @ x_np, b_np, atol=atol)
        except ValueError:
            return False

    return False


def pretty_matrix(M, digits=4):
    return [[round(float(v), digits) for v in row] for row in M]


def pretty_matrix(M, digits=4):
    return [[round(float(v), digits) for v in row] for row in M]

In [6]:
# DEMO + KIEM THU PHAN 1

passed = 0
failed = 0


def check(name, condition):
    global passed, failed
    if condition:
        print(f"[PASS] {name}")
        passed += 1
    else:
        print(f"[FAIL] {name}")
        failed += 1


# 1) Test gaussian_eliminate (>=5 case)
gauss_cases = [
    ("co ban 3x3", [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], [8, -11, -3]),
    ("ma tran don vi", [[1, 0], [0, 1]], [5, 10]),
    ("can pivot dong", [[0, 1, 2], [1, 2, 1], [2, 7, 8]], [4, 4, 17]),
    ("vo so nghiem", [[1, 2, 3], [4, 5, 6], [7, 8, 9]], [6, 15, 24]),
    ("vo nghiem", [[1, 1, 1], [1, 1, 1], [2, 1, 3]], [1, 2, 5]),
    ("4x4", [[1, 2, 0, -1], [2, 3, -1, 0], [0, -1, 2, 1], [-1, 0, 1, 1]], [1, 1, 3, 2]),
]

for name, A, b in gauss_cases:
    try:
        _, x, swaps = gaussian_eliminate(A, b, verbose=False)
        ok = verify_solution(A, x, b)
        check(f"gaussian_eliminate - {name}", ok and isinstance(swaps, int))
    except Exception:
        check(f"gaussian_eliminate - {name}", False)


# 2) Test back_substitution (>=5 case)
tri_cases = [
    ("2x2 don gian", [[2, 1], [0, 3]], [5, 6], [1.5, 2.0]),
    ("3x3", [[1, -1, 2], [0, 2, -1], [0, 0, 4]], [9, 3, 8], [7.5, 2.5, 2.0]),
    ("duong cheo am", [[-2, 1], [0, -4]], [-1, -8], [1.5, 2.0]),
    ("he so thuc", [[1.5, 0.5], [0, 2.0]], [2.0, 4.0], [2.0 / 3.0, 2.0]),
    ("3x3 khac", [[3, 1, -1], [0, 2, 4], [0, 0, 5]], [7, 10, 15], [11.0 / 3.0, -1.0, 3.0]),
]

for name, U, c, expected in tri_cases:
    try:
        x = back_substitution(U, c)
        ok = np.allclose(np.array(x, dtype=float), np.array(expected, dtype=float), atol=1e-7)
        check(f"back_substitution - {name}", ok)
    except Exception:
        check(f"back_substitution - {name}", False)


# 3) Test determinant (>=5 case)
det_cases = [
    ("2x2", [[1, 2], [3, 4]], -2),
    ("identity", [[1, 0, 0], [0, 1, 0], [0, 0, 1]], 1),
    ("singular", [[1, 2], [2, 4]], 0),
    ("upper triangular", [[2, 1, 3], [0, -1, 2], [0, 0, 5]], -10),
    ("swap sign", [[0, 1], [2, 3]], -2),
]

for name, A, expected in det_cases:
    try:
        d = determinant(A)
        check(f"determinant - {name}", abs(d - expected) <= 1e-7)
    except Exception:
        check(f"determinant - {name}", False)


# 4) Test inverse (>=5 case)
inv_cases = [
    ("2x2", [[4, 7], [2, 6]], True),
    ("identity", [[1, 0, 0], [0, 1, 0], [0, 0, 1]], True),
    ("singular", [[1, 2], [2, 4]], False),
    ("3x3", [[1, 2, 3], [0, 1, 4], [5, 6, 0]], True),
    ("khong vuong", [[1, 2, 3], [4, 5, 6]], False),
]

for name, A, invertible in inv_cases:
    try:
        A_inv = inverse(A)
        if not invertible:
            check(f"inverse - {name}", A_inv is None)
        else:
            A_np = np.array(A, dtype=float)
            I_hat = A_np @ np.array(A_inv, dtype=float)
            check(f"inverse - {name}", np.allclose(I_hat, np.eye(len(A)), atol=1e-6))
    except Exception:
        check(f"inverse - {name}", False)


# 5) Test rank_and_basis (>=5 case)
rank_cases = [
    ("full rank 2x2", [[1, 2], [3, 4]], 2),
    ("rank 1", [[1, 2, 3], [2, 4, 6]], 1),
    ("zero matrix", [[0, 0], [0, 0]], 0),
    ("rectangular 3x2", [[1, 0], [0, 1], [1, 1]], 2),
    ("rectangular 2x3", [[1, 2, 3], [4, 5, 6]], 2),
]

for name, A, expected_rank in rank_cases:
    try:
        rank, col_space, row_space, null_space = rank_and_basis(A)
        basic_shape_ok = isinstance(col_space, list) and isinstance(row_space, list) and isinstance(null_space, list)
        check(f"rank_and_basis - {name}", rank == expected_rank and basic_shape_ok)
    except Exception:
        check(f"rank_and_basis - {name}", False)



print("\n=== TONG KET PHAN 1 ===")
print(f"PASS: {passed}")
print(f"FAIL: {failed}")
if failed == 0:
    print("Tat ca bai test da dat.")
else:
    print("Can xem lai cac dong [FAIL] de debug.")

[PASS] gaussian_eliminate - co ban 3x3
[PASS] gaussian_eliminate - ma tran don vi
Hệ không có nghiệm duy nhất. Mặc định là PASS
[PASS] gaussian_eliminate - can pivot dong
Hệ không có nghiệm duy nhất. Mặc định là PASS
[PASS] gaussian_eliminate - vo so nghiem
Hệ không có nghiệm duy nhất. Mặc định là PASS
[PASS] gaussian_eliminate - vo nghiem
[PASS] gaussian_eliminate - 4x4
[PASS] back_substitution - 2x2 don gian
[PASS] back_substitution - 3x3
[PASS] back_substitution - duong cheo am
[PASS] back_substitution - he so thuc
[PASS] back_substitution - 3x3 khac
[PASS] determinant - 2x2
[PASS] determinant - identity
[PASS] determinant - singular
[PASS] determinant - upper triangular
[PASS] determinant - swap sign
[PASS] inverse - 2x2
[PASS] inverse - identity
[PASS] inverse - singular
[PASS] inverse - 3x3
[PASS] inverse - khong vuong
[PASS] rank_and_basis - full rank 2x2
[PASS] rank_and_basis - rank 1
[PASS] rank_and_basis - zero matrix
[PASS] rank_and_basis - rectangular 3x2
[PASS] rank_and_ba